# Lab 5: Quantum teleportation

**Goal:** move an unknown qubit state from Alice to Bob using one shared
Bell pair and two classical bits.

Nothing travels faster than light: Bob's qubit is useless until the two
classical bits arrive. And the original is destroyed, so the state is
moved, not copied.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/solutions/05_teleportation.ipynb)

**Solution notebook.**

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run(qc, shots=1000, backend=sim):
    """Run a circuit and return the measurement counts."""
    return backend.run(transpile(qc, backend), shots=shots).result().get_counts()

from qiskit import ClassicalRegister, QuantumRegister
import numpy as np

## 5.1 The state to send

Alice's qubit gets an arbitrary state from two rotations. Later we undo
these rotations on Bob's qubit: if teleportation worked, Bob always
measures 0.

In [ ]:
theta, phi = 1.1, 0.7   # any values work; try your own

def prepare(qc, q):
    qc.ry(theta, q)
    qc.rz(phi, q)

def unprepare(qc, q):
    qc.rz(-phi, q)
    qc.ry(-theta, q)

## 5.2 The protocol

| Step | Who | What |
|---|---|---|
| 1 | Both | Share a Bell pair: qubit 1 for Alice, qubit 2 for Bob |
| 2 | Alice | CNOT from her message qubit (0) to qubit 1, then H on qubit 0 |
| 3 | Alice | Measure qubits 0 and 1, send the two bits to Bob |
| 4 | Bob | If bit 1 is 1, apply X. If bit 0 is 1, apply Z |

`qc.if_test(...)` applies a gate only when a measured bit has a given
value, the way Bob reacts to Alice's message.

In [ ]:
msg, alice, bob = QuantumRegister(1, "msg"), QuantumRegister(1, "alice"), QuantumRegister(1, "bob")
c_alice = ClassicalRegister(2, "alice_bits")
c_bob = ClassicalRegister(1, "bob_result")
qc = QuantumCircuit(msg, alice, bob, c_alice, c_bob)

prepare(qc, msg[0])
qc.barrier()

# Step 1: Bell pair between alice and bob
qc.h(alice[0])
qc.cx(alice[0], bob[0])
qc.barrier()

# Step 2-3: Alice's operations and measurements
qc.cx(msg[0], alice[0])
qc.h(msg[0])
qc.measure(msg[0], c_alice[0])
qc.measure(alice[0], c_alice[1])
qc.barrier()

# Step 4: Bob's corrections
with qc.if_test((c_alice[1], 1)):
    qc.x(bob[0])
with qc.if_test((c_alice[0], 1)):
    qc.z(bob[0])

# Check: undo the preparation, Bob should always see 0
unprepare(qc, bob[0])
qc.measure(bob[0], c_bob[0])

display(qc.draw("mpl"))
counts = run(qc, shots=2000)
print(counts)

## 5.3 Reading the results

Each key reads `bob_result alice_bits`. Alice's two bits are random
(all four combinations appear), but Bob's bit, the leftmost, should be 0
in every shot.

In [ ]:
bob_ones = sum(n for key, n in counts.items() if key.split()[0] == "1")
print("Shots where Bob measured 1:", bob_ones)
assert bob_ones == 0

## Exercise 5.1: Skip the corrections

Comment out Bob's corrections and run again. How often does Bob measure
1 now? What does that say about using entanglement without the classical
message?

## Summary

- Teleportation moves a state using entanglement plus two classical bits.
- Without the classical bits, Bob's qubit is random noise, so no
  information travels faster than light.
- The same idea underlies quantum networks and some error-correction
  schemes.

Finished early? Try the optional BB84 notebook.